In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed


In [ ]:
file='DataExport132110.csv'

In [ ]:
def load_data(file):
    path = f'processed_data/{file}'
    df = pd.read_csv(path, parse_dates=['timestamp'], index_col='timestamp')
    df.index = pd.to_datetime(df.index).sort_values()
    return df.loc[:, ~df.columns.str.contains('^Unnamed')]

### Hyperparamters

In [ ]:
# Define hyperparameters
HYPERPARAMETERS = {
    'window_size': 96,  # 24 hours
    'min_pct_change': 1.0,
    'min_duration': 96 * 1  # 7 days
}

### Functions

In [ ]:
# Core detection function
def detect_increasing_trend(series, **params):
    series = series.interpolate().fillna(method='bfill').fillna(method='ffill')
    rolling_mean = series.rolling(window=params['window_size'], center=True).mean()
    pct_changes = rolling_mean.pct_change() * 100
    
    increasing_periods = []
    start_idx = None
    current_trend = 0
    
    for idx in range(1, len(series)):
        if pct_changes[idx] > 0:
            start_idx = idx if start_idx is None else start_idx
            current_trend += pct_changes[idx]
        elif start_idx is not None:
            duration = idx - start_idx
            if duration >= params['min_duration'] and current_trend >= params['min_pct_change']:
                increasing_periods.append({
                    'start_time': series.index[start_idx],
                    'end_time': series.index[idx],
                    'total_increase_pct': round(current_trend, 2),
                    'duration': duration
                })
            start_idx = None
            current_trend = 0
    
    return increasing_periods

# Analysis and visualization functions
def analyze_meter(name, series):
    results = detect_increasing_trend(series, **HYPERPARAMETERS)
    return {'meter': name, 'has_change': bool(results), 'changes': results}

def plot_meter_with_changes(series, changes, title=''):
    plt.figure(figsize=(15, 6))
    plt.plot(series.index, series.values, label='Original Data', alpha=0.5)
    plt.plot(series.rolling(window=96, center=True).mean(), 
             label='Rolling Mean', linewidth=2)
    
    for period in changes:
        plt.axvspan(period['start_time'], period['end_time'],
                    alpha=0.2, color='red',
                    label=f'Increase: {period["total_increase_pct"]}%')
    
    plt.title(title)
    plt.xlabel('Time')
    plt.ylabel('Value')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Main execution
def main(file):
    df = load_data(file)
    results = Parallel(n_jobs=-1)(delayed(analyze_meter)(name, df[name]) 
                                 for name in df.columns)
    
    df_results = pd.DataFrame([r for r in results if r['has_change']])
    print(f"Found changes in {len(df_results)} meters")
    
    return df, results, df_results

In [ ]:
df, results, df_results=main(file)

In [ ]:
# Cell 5: Create results dataframe
print(f"Found changes in {len(df_results)} meters")

In [ ]:
# Cell 7: Plot results for meters with changes
for result in results:
    if result['has_change']:
        meter = result['meter']
        print(f"\nAnalyzing {meter}:")
        for change in result['changes']:
            print(f"Increase of {change['total_increase_pct']}% ")
            print(f"From: {change['start_time']}")
            print(f"To: {change['end_time']}")
        
        plot_meter_with_changes(df[meter], result['changes'], title=meter)
        plt.pause(0.5)


# other

In [ ]:
# # Cell 8: Analyze specific meter
# def analyze_specific_meter(meter_name):
#     result = next(r for r in results if r['meter'] == meter_name)
#     plot_meter_with_changes(df[meter_name], result['changes'], title=meter_name)
#     return result

# # Example usage:
# meter = "meter1"  # Replace with your meter of interest
# result = analyze_specific_meter(meter)